In [6]:
# Step 1: Data Exploration

import pandas as pd

# Read the data (file path must be data/diabetic_data.csv)
df = pd.read_csv('data/diabetic_data.csv')

# Print the size of the data to the screen
print(f"Dataset Size (Row, Column): {df.shape}")

# Show the first 5 lines
df.head()

Dataset Size (Row, Column): (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [7]:
# STEP 2: DATA CLEANING AND TRANSFORMATION 

# converting the '?' symbols to actual NaN (Missing Data) values.
import numpy as np
df = df.replace('?', np.nan)

# to see how much missing data there is in each column

print("--- Missing Data Rates (%) ---")
missing_percentages = (df.isnull().sum() / len(df)) * 100
print(missing_percentages[missing_percentages > 0].sort_values(ascending=False))
print("\n")

# Remove columns with too much missing data.# 'weight' (%96 missing), 'medical_specialty' (%49 missing) and 'payer_code' (%39 missing)
# Deleting these columns because they would mislead our model.
df = df.drop(['weight', 'medical_specialty', 'payer_code'], axis=1)

# Also, patient IDs cannot be a feature in the model; excluding them as well.
df = df.drop(['encounter_id', 'patient_nbr'], axis=1)

# convert the Age column to numerical values ​​(average).
# '[0-10)' -> 5, '[10-20)' -> 15 
age_mapping = {'[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, 
               '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, 
               '[80-90)': 85, '[90-100)': 95}
df['age'] = df['age'].replace(age_mapping)

# 4. CREATING Target Variable 
# The 'readmitted' column contains: 'NO', '>30' (patients readmitted after 30 days), and '<30' (patients readmitted within 30 days).
# What we are looking for (the situation where we would be penalized) is '<30'.
# If it is '<30', it will be 1 (High Risk), otherwise 0.
df['target_readmitted_under_30'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

# Delete the original 'readmitted' column.
df = df.drop(['readmitted'], axis=1)

# Fill the remaining missing values (race, diag_1 etc.) with the most frequent (mode) values
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Data Cleaning Completed!")
print(f"New Shape: {df.shape}")
print("Target Variable Distribution:")
print(df['target_readmitted_under_30'].value_counts())

--- Missing Data Rates (%) ---
weight               96.858479
max_glu_serum        94.746772
A1Cresult            83.277322
medical_specialty    49.082208
payer_code           39.557416
race                  2.233555
diag_3                1.398306
diag_2                0.351787
diag_1                0.020636
dtype: float64


Data Cleaning Completed!
New Shape: (101766, 45)
Target Variable Distribution:
target_readmitted_under_30
0    90409
1    11357
Name: count, dtype: int64


In [8]:
# STEP 3: ADVANCED FEATURE ENGINEERING (ICD-9 and Encoding) 

# 1. Function to Group ICD-9 Codes (for diag_1, diag_2, diag_3)
# reduce diseases from hundreds of specific codes into 9 main categories.
def map_icd9(val):
    if str(val).startswith('V') or str(val).startswith('E'):
        return 'Other'
    try:
        num = float(val)
        if 390 <= num <= 459 or num == 785:
            return 'Circulatory' # Circulatory System
        elif 460 <= num <= 519 or num == 786:
            return 'Respiratory' # Respiratory System
        elif 520 <= num <= 579 or num == 787:
            return 'Digestive'   # Digestive System
        elif np.floor(num) == 250:
            return 'Diabetes'    # Diabetes
        elif 800 <= num <= 999:
            return 'Injury'      # Injury
        elif 710 <= num <= 739:
            return 'Musculoskeletal' # Musculoskeletal System
        elif 580 <= num <= 629 or num == 788:
            return 'Genitourinary' # Genitourinary System
        elif 140 <= num <= 239:
            return 'Neoplasms'   # Tumor/Cancer
        else:
            return 'Other'       # Other
    except:
        return 'Other'

# Let's apply the function to all 3 diagnosis columns
for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(map_icd9)

# 2. Converting Categorical Variables into Numerical (One-Hot Encoding)
# Machine learning models cannot understand text like 'Female' or 'Circulatory'.
# We need to convert them into new columns consisting of 0s and 1s (True/False).
categorical_cols = df.select_dtypes(include=['object', 'string']).columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Observation
print("Encoding has completed!")
print(f"Ready Data Shape for Model: {df_encoded.shape}")

Encoding has completed!
Ready Data Shape for Model: (101766, 105)


In [9]:
#  STEP 4: MACHINE LEARNING MODELING 
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_recall_curve, auc

# 1. Separating Features (X) and Target Variable (y)
X = df_encoded.drop('target_readmitted_under_30', axis=1)
y = df_encoded['target_readmitted_under_30']

# 2. Splitting the Data into Training (80%) and Test (20%)
# The stratify=y parameter ensures that the 11% imbalanced distribution is preserved in both training and test sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

# 3. Defining the Model (we use class_weight='balanced' for imbalanced data)
# Random Forest is a powerful tree-based model. n_jobs=-1 allows it to use all CPU cores.
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)

# 4. Training the Model
print("Model has been trained.")
rf_model.fit(X_train, y_train)

# 5. Predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# 6. Evaluation Metrics
print("\n-- Model Evaluation Report")
print(classification_report(y_test, y_pred))

# Calculating PR-AUC (Precision-Recall Area Under Curve)
# This metric is much more reliable than ROC-AUC for imbalanced datasets.
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
pr_auc = auc(recall, precision)
print(f"PR-AUC Score: {pr_auc:.3f}")

Training set size: (81412, 104)
Test set size: (20354, 104)
Model has been trained.

-- Model Evaluation Report
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18083
           1       0.53      0.00      0.01      2271

    accuracy                           0.89     20354
   macro avg       0.71      0.50      0.47     20354
weighted avg       0.85      0.89      0.84     20354

PR-AUC Score: 0.206


In [10]:
#  STEP 5: VALUE-BASED COST OPTIMIZATION 
import numpy as np

# 1. Cost Parameters (in Euros)
COST_PENALTY = 10000  # VBR penalty if we miss a patient (False Negative)
COST_INTERVENTION = 500 # Cost of preventive intervention (False Positive / True Positive)

# 2. What if we had no model? (Baseline)
# We do not intervene for any patient, and we pay a penalty for everyone who is readmitted.
actual_readmissions = np.sum(y_test)
baseline_cost = actual_readmissions * COST_PENALTY
print(f"Total penalty if we had not used any model: {baseline_cost:,.0f} €")


# 3. Finding the Best Threshold for Profit/Loss Optimization
thresholds = np.arange(0.05, 0.55, 0.05) # We will test thresholds from 5% to 50%
best_savings = 0
best_threshold = 0.5
best_preds = None

print("\n--- Cost Analysis at Different Risk Thresholds ---")
for t in thresholds:
    # Convert predicted probabilities into 1 or 0 based on our custom threshold
    custom_preds = (y_pred_proba >= t).astype(int)
    
    # Confusion Matrix Components
    TP = np.sum((custom_preds == 1) & (y_test == 1)) # Correct prediction, we intervened
    FP = np.sum((custom_preds == 1) & (y_test == 0)) # Wrong prediction, unnecessary intervention
    FN = np.sum((custom_preds == 0) & (y_test == 1)) # Missed case, we got penalized!
    TN = np.sum((custom_preds == 0) & (y_test == 0)) # Correct prediction, no intervention. Cost = 0.
    
    # Total Cost Calculation
    total_cost = (TP * COST_INTERVENTION) + (FP * COST_INTERVENTION) + (FN * COST_PENALTY)
    
    # How much did we save?
    savings = baseline_cost - total_cost
    print(f"Threshold: {t:.2f} -> Remaining Cost/Penalty: {total_cost:,.0f} € | SAVINGS: {savings:,.0f} €")

    
    if savings > best_savings:
        best_savings = savings
        best_threshold = t
        best_preds = custom_preds

print("\n" + "="*50)
print(f" BEST BUSINESS DECISION (OPTIMUM THRESHOLD): {best_threshold:.2f}")
print(f" NET PROFIT FOR THE HOSPITAL: {best_savings:,.0f} €")
print("="*50)

# 4. Exporting Results for Power BI
# Since we do not retain the original IDs of patients in the test set,
# create a new DataFrame containing risk scores for visualization purposes.
results_df = pd.DataFrame({
    'Real situation': y_test,
    'Risk Probability': np.round(y_pred_proba * 100, 2),
    'Model_Decision': best_preds
})

results_df.to_csv('data/dashboard_data.csv', index=False)
print("\nThe 'dashboard_data.csv' file for Power BI/Tableau has been successfully created!")


Total penalty if we had not used any model: 22,710,000 €

--- Cost Analysis at Different Risk Thresholds ---
Threshold: 0.05 -> Remaining Cost/Penalty: 10,342,000 € | SAVINGS: 12,368,000 €
Threshold: 0.10 -> Remaining Cost/Penalty: 12,005,500 € | SAVINGS: 10,704,500 €
Threshold: 0.15 -> Remaining Cost/Penalty: 16,335,500 € | SAVINGS: 6,374,500 €
Threshold: 0.20 -> Remaining Cost/Penalty: 18,672,500 € | SAVINGS: 4,037,500 €
Threshold: 0.25 -> Remaining Cost/Penalty: 20,585,000 € | SAVINGS: 2,125,000 €
Threshold: 0.30 -> Remaining Cost/Penalty: 21,558,500 € | SAVINGS: 1,151,500 €
Threshold: 0.35 -> Remaining Cost/Penalty: 22,161,500 € | SAVINGS: 548,500 €
Threshold: 0.40 -> Remaining Cost/Penalty: 22,393,000 € | SAVINGS: 317,000 €
Threshold: 0.45 -> Remaining Cost/Penalty: 22,546,500 € | SAVINGS: 163,500 €
Threshold: 0.50 -> Remaining Cost/Penalty: 22,610,000 € | SAVINGS: 100,000 €

 BEST BUSINESS DECISION (OPTIMUM THRESHOLD): 0.05
 NET PROFIT FOR THE HOSPITAL: 12,368,000 €

The 'dashboa